# Lab type: debug
# Course: ML301 — Deep Learning with PyTorch
# Lesson: Convolutional Neural Networks
# Task: The CNN code below contains 3 bugs. All run without errors but produce wrong results. Find each bug, explain it in the comment cell below it, and fix it.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)

# Synthetic image dataset: 500 images, 3 channels, 32x32 pixels, 10 classes
n_samples = 500
X = torch.randn(n_samples, 3, 32, 32)
y = torch.randint(0, 10, (n_samples,))

print(f"Dataset: {n_samples} images, shape {X[0].shape}, {y.unique().numel()} classes")

## Spatial Dimension Reference

For `nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p)`:
```
H_out = floor((H_in + 2*padding - kernel_size) / stride) + 1
```
For `nn.MaxPool2d(kernel_size=2, stride=2)`:
```
H_out = floor(H_in / 2)
```
Use this to verify `in_features` for the first Linear layer.

## Bug 1: Hardcoded in_features

The CNN below computes the wrong `in_features` for the first Linear layer. The architecture uses: conv1 (3→16, k=3, p=1) → maxpool → conv2 (16→32, k=3, p=1) → maxpool → flatten → linear.

For a 32×32 input: after conv1+pool: 16×16; after conv2+pool: 8×8; so in_features should be 32×8×8 = 2048. The bug uses 512.

In [ ]:
# --- BUGGY CODE (Bug 1) ---
class BuggyNet1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        # BUG: in_features is 512, but actual flattened size is 32 * 8 * 8 = 2048
        self.fc1 = nn.Linear(512, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

buggy_net1 = BuggyNet1()

try:
    out = buggy_net1(X[:4])
    print(f"Output shape: {out.shape}")
except RuntimeError as e:
    print(f"RuntimeError: {e}")
    print("\nStep-by-step dimension trace:")
    print("  Input: 3 x 32 x 32")
    print("  After conv1 (k=3, p=1): 16 x 32 x 32")
    print("  After pool (2x2): 16 x 16 x 16")
    print("  After conv2 (k=3, p=1): 32 x 16 x 16")
    print("  After pool (2x2): 32 x 8 x 8")
    print("  Flattened: 32 * 8 * 8 =", 32 * 8 * 8)

**Explain the bug:** Walk through the spatial dimension calculation step by step to show why in_features=512 is wrong. What is the correct value?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**Why in_features=512 is wrong:** After conv1 (k=3, p=1, preserves spatial size) and maxpool(2×2), the feature map is 16×16×16. After conv2 (k=3, p=1) and a second maxpool(2×2), it becomes 32×8×8 = 2048 — not 512. PyTorch raises a `RuntimeError` at the Linear layer because the flattened tensor has 2048 elements but the weight matrix expects 512.

**Correct approach:** Pass `in_features=2048` explicitly, or — better — use a dummy forward pass inside `__init__` with `torch.no_grad()` to compute the flattened size dynamically so the model stays correct if the input resolution or architecture changes.

</details>

In [ ]:
# Fix for Bug 1: compute in_features dynamically using a dummy forward pass
class FixedNet1(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        
        # Dynamic computation — works for any input size
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 32, 32)
            dummy = self.pool(F.relu(self.conv1(dummy)))
            dummy = self.pool(F.relu(self.conv2(dummy)))
            in_features = dummy.view(1, -1).shape[1]
        
        self.fc1 = nn.Linear(in_features, 64)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

fixed_net1 = FixedNet1()
out = fixed_net1(X[:4])
print(f"Output shape: {out.shape}")
print(f"in_features correctly computed: {fixed_net1.fc1.in_features}")

## Bug 2: Training mode active during inference

The evaluation loop calls `model.train()` before iterating the validation set. BatchNorm uses batch statistics instead of running statistics, and Dropout randomly zeroes activations — predictions are non-deterministic.

In [ ]:
# Add BatchNorm to make the bug clearly observable
class NetWithBN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(16 * 16 * 16, 64)
        self.drop = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = x.view(x.size(0), -1)
        x = self.drop(F.relu(self.fc1(x)))
        return self.fc2(x)

net = NetWithBN()

# --- BUGGY CODE (Bug 2) ---
# BUG: model.train() called before inference — should be model.eval()
net.train()

run1 = net(X[:8]).argmax(dim=1)
run2 = net(X[:8]).argmax(dim=1)

print("Predictions with model in training mode (non-deterministic due to Dropout):")
print(f"  Run 1: {run1.tolist()}")
print(f"  Run 2: {run2.tolist()}")
print(f"  Identical: {(run1 == run2).all().item()}")

**Explain the bug:** Which two layer types behave differently in training vs evaluation mode, and how does each misbehave during inference?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**How training mode breaks inference:** Two layers behave differently depending on mode. (1) `BatchNorm` — in training mode it normalises using the current batch's mean/std; in eval mode it uses the running statistics accumulated during training. (2) `Dropout` — in training mode it randomly zeroes activations with probability `p`; in eval mode it passes all activations through unchanged. Calling `model.train()` before inference makes predictions non-deterministic: two identical forward passes over the same data return different results.

**Correct approach:** Always call `model.eval()` (and `torch.no_grad()`) before any inference or evaluation loop. Restore `model.train()` only when resuming gradient-based training.

</details>

In [ ]:
# Fix: set evaluation mode before inference
# This disables Dropout and switches BatchNorm to use running statistics
net_fix = NetWithBN()
net_fix.eval()  # Set evaluation mode

with torch.no_grad():
    run1_fix = net_fix(X[:8]).argmax(dim=1)
    run2_fix = net_fix(X[:8]).argmax(dim=1)

print("Predictions with model in evaluation mode (deterministic):")
print(f"  Run 1: {run1_fix.tolist()}")
print(f"  Run 2: {run2_fix.tolist()}")
print(f"  Identical: {(run1_fix == run2_fix).all().item()}")

## Bug 3: Missing batch dimension

A single image is passed to the network as a 3D tensor `(C, H, W)`. Conv2d expects 4D input `(B, C, H, W)`. The network misinterprets the channel dimension as the batch dimension.

In [ ]:
# A single preprocessed image — shape (3, 32, 32)
single_image = X[0]
print(f"single_image.shape: {single_image.shape}")

# --- BUGGY CODE (Bug 3) ---
net_bug3 = FixedNet1()
net_bug3.eval()

try:
    with torch.no_grad():
        # BUG: Conv2d expects (B, C, H, W) but receives (C, H, W) = (3, 32, 32)
        # Interpreted as: batch=3, channels=32, H=32, W=missing
        out = net_bug3(single_image)
    print(f"Output shape: {out.shape}")
except RuntimeError as e:
    print(f"RuntimeError (expected): {e}")
    print("\nConv2d interpreted single_image as (batch=3, channels=32, H=32)")
    print("The channel count 3 doesn't match conv1's expected in_channels=3 for a 4D input")

**Explain the bug:** What does Conv2d interpret the 3D tensor dimensions as? What is the correct fix?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What Conv2d interprets:** `nn.Conv2d` requires 4D input `(B, C, H, W)`. Passing the 3D tensor `(3, 32, 32)` makes Conv2d interpret the dimensions as `(batch=3, channels=32, H=32)` — the channel axis is read as batch size and the spatial width dimension is missing entirely, causing a `RuntimeError`.

**Correct approach:** Add a batch dimension with `single_image.unsqueeze(0)`, converting `(3, 32, 32)` → `(1, 3, 32, 32)` before passing to the network. For production inference pipelines, this is usually handled automatically by the `DataLoader` collate function.

</details>

In [ ]:
# Fix: add batch dimension with unsqueeze(0)
single_image_4d = single_image.unsqueeze(0)  # (3, 32, 32) -> (1, 3, 32, 32)
print(f"After unsqueeze(0): {single_image_4d.shape}")

net_bug3.eval()
with torch.no_grad():
    out = net_bug3(single_image_4d)
print(f"Output shape: {out.shape}")
print(f"Predicted class: {out.argmax(dim=1).item()}")

## Summary

> **For each bug, write one sentence on what went wrong and how to prevent it.**

1. 
2. 
3. 

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Bug 1 — hardcoded in_features:** The flattened feature size after two conv+pool blocks is 32×8×8=2048, not 512; hardcoding the wrong value causes a shape mismatch — compute it dynamically with a dummy forward pass.

2. **Bug 2 — training mode at inference:** Calling `model.train()` before evaluation leaves Dropout active and forces BatchNorm to use batch statistics, making predictions non-deterministic — always call `model.eval()` before any inference loop.

3. **Bug 3 — missing batch dimension:** Conv2d requires 4D `(B, C, H, W)` input; passing a 3D `(C, H, W)` single image causes a shape error — use `unsqueeze(0)` to prepend the batch dimension.

</details>